# Pharmacy Database Management System
This notebook represents the entire Pharmacy Database Management System project in a single, interactive format.
It demonstrates database normalization (3NF), relational logic, analytical SQL queries, and code architecture as required for CS 210.

## Overview
1. **Database Connection & Setup**: Initialize the SQLite database.
2. **Schema Creation**: Create tables for `location`, `pharmacy`, `medicine`, `customer`, `available`, and `purchase`.
3. **Data Management**: Functions to add, display, and delete data.
4. **Demo Data**: Populate the database with sample data.
5. **Analytics & Reports**: Perform SQL queries to generate insights.


In [1]:
import sqlite3
import os
import sys
import io

DB_PATH = 'pharmacy_database.db'

def get_connection():
    """Create and return a database connection"""
    return sqlite3.connect(DB_PATH)

def show_database_info():
    """Display database connection and schema information"""
    print("\n" + "="*70)
    print("DATABASE CONNECTION INFORMATION")
    print("="*70)
    print(f"Database Type: SQLite")
    print(f"Database File: {DB_PATH}")
    print(f"Full Path: {os.path.abspath(DB_PATH)}")
    print("="*70)

show_database_info()



DATABASE CONNECTION INFORMATION
Database Type: SQLite
Database File: pharmacy_database.db
Full Path: C:\Users\Vasan\Documents\Project\Pharma\Pharmacy Database Mangement System Using Python and SQL\pharmacy_database.db


## Database Initialization
Here we define the schema. The database is in 3NF, ensuring data integrity and minimizing redundancy.


In [2]:
def init_database():
    """Initialize the database with all tables"""
    conn = get_connection()
    cursor = conn.cursor()
    
    # Ensure fresh schema for CodeBench compatibility
    cursor.execute("DROP TABLE IF EXISTS purchase")
    cursor.execute("DROP TABLE IF EXISTS available")
    cursor.execute("DROP TABLE IF EXISTS customer")
    cursor.execute("DROP TABLE IF EXISTS medicine")
    cursor.execute("DROP TABLE IF EXISTS pharmacy")
    cursor.execute("DROP TABLE IF EXISTS location")

    # Location table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS location (
            location_id TEXT PRIMARY KEY,
            location_name TEXT NOT NULL
        )
    ''')
    
    # Pharmacy table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS pharmacy (
            pharmacy_id TEXT PRIMARY KEY,
            pharmacy_name TEXT NOT NULL,
            location_id TEXT,
            FOREIGN KEY (location_id) REFERENCES location(location_id)
        )
    ''')
    
    # Medicine table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS medicine (
            med_id TEXT PRIMARY KEY,
            med_name TEXT NOT NULL,
            cost REAL NOT NULL,
            dosage_mg INTEGER NOT NULL
        )
    ''')
    
    # Customer table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS customer (
            cust_id TEXT PRIMARY KEY,
            name TEXT NOT NULL,
            age INTEGER,
            address TEXT,
            phone TEXT,
            pharmacy_id TEXT,
            FOREIGN KEY (pharmacy_id) REFERENCES pharmacy(pharmacy_id)
        )
    ''')
    
    # Available table (junction table for medicine-pharmacy)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS available (
            med_id TEXT,
            pharmacy_id TEXT,
            stock_quantity INTEGER DEFAULT 50,
            PRIMARY KEY (med_id, pharmacy_id),
            FOREIGN KEY (med_id) REFERENCES medicine(med_id),
            FOREIGN KEY (pharmacy_id) REFERENCES pharmacy(pharmacy_id)
        )
    ''')
    
    # Purchase table (junction table for medicine-customer)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS purchase (
            purchase_id INTEGER PRIMARY KEY AUTOINCREMENT,
            med_id TEXT,
            cust_id TEXT,
            purchase_date TEXT,
            quantity INTEGER DEFAULT 1,
            total_cost REAL,
            FOREIGN KEY (med_id) REFERENCES medicine(med_id),
            FOREIGN KEY (cust_id) REFERENCES customer(cust_id)
        )
    ''')
    
    conn.commit()
    conn.close()
    print("✓ Database initialized successfully!")

# Initialize the tables
init_database()


✓ Database initialized successfully!

## Data Management Functions
Functions to insert and view data across the tables.


In [3]:
def add_location(location_id, location_name):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO location VALUES (?, ?)", (location_id, location_name))
        conn.commit()
        print(f"✓ Location '{location_name}' added successfully!")
    except sqlite3.IntegrityError:
        print(f"✗ Error: Location ID '{location_id}' already exists!")
    finally:
        conn.close()

def add_pharmacy(pharmacy_id, pharmacy_name, location_id):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO pharmacy VALUES (?, ?, ?)", (pharmacy_id, pharmacy_name, location_id))
        conn.commit()
        print(f"✓ Pharmacy '{pharmacy_name}' added successfully!")
    except sqlite3.IntegrityError as e:
        print(f"✗ Error: {e}")
    finally:
        conn.close()

def add_medicine(med_id, med_name, cost, dosage_mg):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO medicine VALUES (?, ?, ?, ?)", (med_id, med_name, float(cost), int(dosage_mg)))
        conn.commit()
        print(f"✓ Medicine '{med_name}' added successfully!")
    except sqlite3.IntegrityError:
        print(f"✗ Error: Medicine ID '{med_id}' already exists!")
    finally:
        conn.close()

def add_customer(cust_id, name, age, address, phone, pharmacy_id):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO customer VALUES (?, ?, ?, ?, ?, ?)", (cust_id, name, int(age), address, phone, pharmacy_id))
        conn.commit()
        print(f"✓ Customer '{name}' added successfully!")
    except sqlite3.IntegrityError:
        print(f"✗ Error: Customer ID '{cust_id}' already exists!")
    finally:
        conn.close()

def add_available(med_id, pharmacy_id, stock_quantity=50):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO available (med_id, pharmacy_id, stock_quantity) VALUES (?, ?, ?)", (med_id, pharmacy_id, stock_quantity))
        conn.commit()
        print(f"✓ Medicine '{med_id}' is now available at pharmacy '{pharmacy_id}'!")
    except sqlite3.IntegrityError:
        print(f"✗ Error: This availability record already exists!")
    finally:
        conn.close()

def add_purchase(med_id, cust_id, quantity=1):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        # Look up cost for total_cost calculation
        cursor.execute("SELECT cost FROM medicine WHERE med_id = ?", (med_id,))
        row = cursor.fetchone()
        cost = row[0] if row else 0
        total_cost = cost * quantity
        from datetime import date
        cursor.execute("INSERT INTO purchase (med_id, cust_id, quantity, total_cost, purchase_date) VALUES (?, ?, ?, ?, ?)", (med_id, cust_id, quantity, total_cost, date.today().isoformat()))
        conn.commit()
        print(f"✓ Purchase recorded: Medicine '{med_id}' by Customer '{cust_id}'!")
    except sqlite3.IntegrityError:
        print(f"✗ Error: This purchase record already exists!")
    finally:
        conn.close()


## Populating Demo Data
Let's populate our database with sample data so we can perform queries on it.


In [4]:
def run_demo():
    print("=" * 60)
    print("PHARMACY DATABASE MANAGEMENT SYSTEM - DEMO")
    print("=" * 60)
    
    print("\n1. ADDING LOCATIONS")
    add_location("LOC001", "New York")
    add_location("LOC002", "Los Angeles")
    add_location("LOC003", "Chicago")
    
    print("\n2. ADDING PHARMACIES")
    add_pharmacy("PH001", "City Pharmacy", "LOC001")
    add_pharmacy("PH002", "Health Plus", "LOC002")
    add_pharmacy("PH003", "MediCare", "LOC003")
    
    print("\n3. ADDING MEDICINES")
    add_medicine("MED001", "Paracetamol", 5.99, 500)
    add_medicine("MED002", "Ibuprofen", 7.50, 200)
    add_medicine("MED003", "Amoxicillin", 12.00, 250)
    add_medicine("MED004", "Aspirin", 4.25, 100)
    add_medicine("MED005", "Cetirizine", 8.75, 10)
    
    print("\n4. ADDING CUSTOMERS")
    add_customer("CUST001", "John Doe", 30, "123 Main St, NY", "555-0101", "PH001")
    add_customer("CUST002", "Jane Smith", 25, "456 Oak Ave, LA", "555-0102", "PH002")
    add_customer("CUST003", "Bob Johnson", 45, "789 Pine Rd, Chicago", "555-0103", "PH003")
    
    print("\n5. ADDING MEDICINE AVAILABILITY")
    add_available("MED001", "PH001")
    add_available("MED002", "PH001")
    add_available("MED003", "PH002")
    add_available("MED004", "PH002")
    add_available("MED005", "PH003")
    
    print("\n6. RECORDING PURCHASES")
    add_purchase("MED001", "CUST001")
    add_purchase("MED002", "CUST001")
    add_purchase("MED003", "CUST002")
    add_purchase("MED004", "CUST002")
    add_purchase("MED005", "CUST003")

# Clear tables before demo to avoid IntegrityErrors on multiple runs
conn = get_connection()
cursor = conn.cursor()
for table in ['purchase', 'available', 'customer', 'medicine', 'pharmacy', 'location']:
    cursor.execute(f"DELETE FROM {table}")
conn.commit()
conn.close()

run_demo()


PHARMACY DATABASE MANAGEMENT SYSTEM - DEMO

1. ADDING LOCATIONS
✓ Location 'New York' added successfully!
✓ Location 'Los Angeles' added successfully!
✓ Location 'Chicago' added successfully!

2. ADDING PHARMACIES
✓ Pharmacy 'City Pharmacy' added successfully!
✓ Pharmacy 'Health Plus' added successfully!
✓ Pharmacy 'MediCare' added successfully!

3. ADDING MEDICINES
✓ Medicine 'Paracetamol' added successfully!
✓ Medicine 'Ibuprofen' added successfully!
✓ Medicine 'Amoxicillin' added successfully!
✓ Medicine 'Aspirin' added successfully!
✓ Medicine 'Cetirizine' added successfully!

4. ADDING CUSTOMERS
✓ Customer 'John Doe' added successfully!
✓ Customer 'Jane Smith' added successfully!
✓ Customer 'Bob Johnson' added successfully!

5. ADDING MEDICINE AVAILABILITY
✓ Medicine 'MED001' is now available at pharmacy 'PH001'!
✓ Medicine 'MED002' is now available at pharmacy 'PH001'!
✓ Medicine 'MED003' is now available at pharmacy 'PH002'!
✓ Medicine 'MED004' is now available at pharmacy 'PH00

## Viewing Data using SQL Queries
Using Pandas to display the SQL output formats the data nicely as HTML tables.


In [5]:
import pandas as pd

def query_to_df(query, params=()):
    conn = get_connection()
    df = pd.read_sql_query(query, conn, params=params)
    conn.close()
    return df

# Displaying all pharmacies with their locations
query = '''
    SELECT p.pharmacy_id, p.pharmacy_name, l.location_name 
    FROM pharmacy p
    LEFT JOIN location l ON p.location_id = l.location_id
'''
display(query_to_df(query))


,pharmacy_id,pharmacy_name,location_name
0,PH001,City Pharmacy,New York
1,PH002,Health Plus,Los Angeles
2,PH003,MediCare,Chicago


## Analytics & Reports
Let's run some advanced queries to generate business insights.


In [6]:
# 1. Total Sales Report
sales_query = '''
    SELECT COUNT(*) as total_purchases, SUM(m.cost) as total_revenue
    FROM purchase p
    JOIN medicine m ON p.med_id = m.med_id
'''
print("Total Sales & Revenue:")
display(query_to_df(sales_query))

# 2. Most Popular Medicines
popular_query = '''
    SELECT m.med_id, m.med_name, COUNT(*) as purchase_count
    FROM purchase p
    JOIN medicine m ON p.med_id = m.med_id
    GROUP BY m.med_id, m.med_name
    ORDER BY purchase_count DESC
'''
print("\nMost Popular Medicines:")
display(query_to_df(popular_query))

# 3. Customer Purchase History
history_query = '''
    SELECT p.med_id, m.med_name, m.cost, m.dosage_mg
    FROM purchase p
    JOIN medicine m ON p.med_id = m.med_id
    WHERE p.cust_id = 'CUST001'
'''
print("\nPurchase History for CUST001:")
display(query_to_df(history_query))


Total Sales & Revenue:


,total_purchases,total_revenue
0,5,38.49



Most Popular Medicines:


,med_id,med_name,purchase_count
0,MED001,Paracetamol,1
1,MED002,Ibuprofen,1
2,MED003,Amoxicillin,1
3,MED004,Aspirin,1
4,MED005,Cetirizine,1



Purchase History for CUST001:


,med_id,med_name,cost,dosage_mg
0,MED001,Paracetamol,5.99,500
1,MED002,Ibuprofen,7.50,200


## Kaggle Dataset (Real Sales Data)
Let's view the real sales data imported from the Kaggle Dataset.

In [7]:
import pandas as pd
import os

def load_kaggle_csv():
    """Load the Kaggle Pharma Sales CSV data into the SQLite database."""
    csv_path = os.path.join('Pharmacy Data', 'salesmonthly.csv')
    if not os.path.exists(csv_path):
        print("Error: CSV file not found at", csv_path)
        return
    
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows from {csv_path}")
    
    conn = get_connection()
    cursor = conn.cursor()
    
    # Medicine definitions based on ATC codes in the Kaggle dataset
    medicines = {
        'M01AB': ('Anti-inflammatory (Acetic)', 15.50, 50),
        'M01AE': ('Anti-inflammatory (Propionic)', 12.00, 200),
        'N02BA': ('Analgesics (Salicylic)', 8.25, 100),
        'N02BE': ('Analgesics (Anilides)', 9.50, 500),
        'N05B':  ('Anxiolytics', 25.00, 10),
        'N05C':  ('Sedatives', 22.50, 5),
        'R03':   ('Airway Disease Drugs', 35.00, 100),
        'R06':   ('Antihistamines', 14.25, 10)
    }
    
    # Insert medicines (skip if already exist)
    for med_id, (name, cost, dosage) in medicines.items():
        try:
            cursor.execute("INSERT INTO medicine VALUES (?, ?, ?, ?)", (med_id, name, cost, dosage))
        except sqlite3.IntegrityError:
            pass
    
    # Create real_sales table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS real_sales (
            sale_id INTEGER PRIMARY KEY AUTOINCREMENT,
            datum TEXT NOT NULL,
            med_id TEXT NOT NULL,
            quantity REAL NOT NULL,
            FOREIGN KEY (med_id) REFERENCES medicine(med_id)
        )
    """)
    cursor.execute("DELETE FROM real_sales")
    
    # Insert sales data
    count = 0
    for _, row in df.iterrows():
        for med_id in medicines:
            qty = row[med_id]
            if qty > 0:
                cursor.execute("INSERT INTO real_sales (datum, med_id, quantity) VALUES (?, ?, ?)",
                               (row['datum'], med_id, float(qty)))
                count += 1
    
    conn.commit()
    conn.close()
    print(f"✓ Inserted {count} Kaggle sales records into real_sales table")

load_kaggle_csv()

Loaded 70 rows from Pharmacy Data\salesmonthly.csv
✓ Inserted 553 Kaggle sales records into real_sales table


In [8]:
"""
This queries the Kaggle dataset sales we imported via load_kaggle_data.py
"""
kaggle_query = '''
    SELECT r.datum as Month, m.med_name as Medicine, r.quantity as Units_Sold
    FROM real_sales r
    JOIN medicine m ON r.med_id = m.med_id
    ORDER BY r.datum DESC
    LIMIT 10
'''
print("Recent Sales Records:")
display(query_to_df(kaggle_query))

Recent Sales Records:


,Month,Medicine,Units_Sold
0,2019-10-31,Anti-inflammatory (Acetic),44.370
1,2019-10-31,Anti-inflammatory (Propionic),37.300
2,2019-10-31,Analgesics (Salicylic),20.650
3,2019-10-31,Analgesics (Anilides),295.150
4,2019-10-31,Anxiolytics,86.000
5,2019-10-31,Sedatives,7.000
6,2019-10-31,Airway Disease Drugs,37.000
7,2019-10-31,Antihistamines,11.130
8,2019-09-30,Anti-inflammatory (Acetic),161.070
9,2019-09-30,Anti-inflammatory (Propionic),111.437
